# Hotel Reservation Predicton

In this project, we will create a model that will predict whether the customer will cancel the reservation of a hotel or keep it.

We are using this dataset from Kaggle : https://www.kaggle.com/datasets/ahsan81/hotel-reservations-classification-dataset

---

# Data Analysis

In [62]:
import numpy as np
import pandas as pd

In [63]:
df = pd.read_csv("data/Hotel Reservations.csv")
df.head()

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,INN00001,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,2017,10,2,Offline,0,0,0,65.00,0,Not_Canceled
1,INN00002,2,0,2,3,Not Selected,0,Room_Type 1,5,2018,11,6,Online,0,0,0,106.68,1,Not_Canceled
2,INN00003,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,2018,2,28,Online,0,0,0,60.00,0,Canceled
3,INN00004,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,2018,5,20,Online,0,0,0,100.00,0,Canceled
4,INN00005,2,0,1,1,Not Selected,0,Room_Type 1,48,2018,4,11,Online,0,0,0,94.50,0,Canceled


In [64]:
print(df.shape)
print(df.columns)

(36275, 19)
Index(['Booking_ID', 'no_of_adults', 'no_of_children', 'no_of_weekend_nights',
       'no_of_week_nights', 'type_of_meal_plan', 'required_car_parking_space',
       'room_type_reserved', 'lead_time', 'arrival_year', 'arrival_month',
       'arrival_date', 'market_segment_type', 'repeated_guest',
       'no_of_previous_cancellations', 'no_of_previous_bookings_not_canceled',
       'avg_price_per_room', 'no_of_special_requests', 'booking_status'],
      dtype='str')


Ok so, we have 36k rows and 19 columns. After looking at the columns, we wouldn't need Booking_ID. And there are different columns for arrival years, months and days. I don't see a use of these either. There is a column lead_time which is the number of days between the date of booking and the arrival date. This might be a factor affecting our result. But I don't see the use the y,m and d. So, I am gonna drop them too. And booking_status is our target column.

## Dropping Unnecessary Columns

In [65]:
df = df.drop(columns= ["Booking_ID", "arrival_year", "arrival_month", "arrival_date"])
df.head(2)

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,Offline,0,0,0,65.00,0,Not_Canceled
1,2,0,2,3,Not Selected,0,Room_Type 1,5,Online,0,0,0,106.68,1,Not_Canceled


In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36275 entries, 0 to 36274
Data columns (total 15 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   no_of_adults                          36275 non-null  int64  
 1   no_of_children                        36275 non-null  int64  
 2   no_of_weekend_nights                  36275 non-null  int64  
 3   no_of_week_nights                     36275 non-null  int64  
 4   type_of_meal_plan                     36275 non-null  str    
 5   required_car_parking_space            36275 non-null  int64  
 6   room_type_reserved                    36275 non-null  str    
 7   lead_time                             36275 non-null  int64  
 8   market_segment_type                   36275 non-null  str    
 9   repeated_guest                        36275 non-null  int64  
 10  no_of_previous_cancellations          36275 non-null  int64  
 11  no_of_previous_bookings_no

Let's analyze other columns.

We don't have any NaN values in the first four cols. 

In [67]:
print(f"Unique values in the type_of_meal_plan col : {df["type_of_meal_plan"].unique().tolist()}")
print(f"Unique values in the required_car_parking_space col : {df["required_car_parking_space"].unique().tolist()}")
print(f"Unique values in the room_type_reserved col : {df["room_type_reserved"].unique().tolist()}")
print(f"Unique values in the market_segment_type col : {df["market_segment_type"].unique().tolist()}")
print(f"Unique values in the repeated_guest col : {df["repeated_guest"].unique().tolist()}")
print(f"Unique values in the no_of_special_requests col : {df["no_of_special_requests"].unique().tolist()}")
print(f"Unique values in the booking_status col : {df["booking_status"].unique().tolist()}")

Unique values in the type_of_meal_plan col : ['Meal Plan 1', 'Not Selected', 'Meal Plan 2', 'Meal Plan 3']
Unique values in the required_car_parking_space col : [0, 1]
Unique values in the room_type_reserved col : ['Room_Type 1', 'Room_Type 4', 'Room_Type 2', 'Room_Type 6', 'Room_Type 5', 'Room_Type 7', 'Room_Type 3']
Unique values in the market_segment_type col : ['Offline', 'Online', 'Corporate', 'Aviation', 'Complementary']
Unique values in the repeated_guest col : [0, 1]
Unique values in the no_of_special_requests col : [0, 1, 3, 2, 4, 5]
Unique values in the booking_status col : ['Not_Canceled', 'Canceled']


We don't have to clean the data much. We can use OHE on type_of_meal_plan, market_segment_type, & room_type_reserved. And we can use mapping to convert the target column to numeric. Else already seems fine.

In [68]:
df["booking_status"] = df["booking_status"].map({"Not_Canceled" : 0, "Canceled" : 1})
df.head(3)

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,Offline,0,0,0,65.00,0,0
1,2,0,2,3,Not Selected,0,Room_Type 1,5,Online,0,0,0,106.68,1,0
2,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,Online,0,0,0,60.00,0,1


We will apply OHE on the remaining after data visualization.

---

## Final Touches

In [69]:
df.duplicated().sum()

np.int64(10746)

There are about 10.7k duplicated rows. Gotta remove them.

In [70]:
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print(df.shape)
df

(25529, 15)


,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,Offline,0,0,0,65.00,0,0
1,2,0,2,3,Not Selected,0,Room_Type 1,5,Online,0,0,0,106.68,1,0
2,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,Online,0,0,0,60.00,0,1
3,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,Online,0,0,0,100.00,0,1
4,2,0,1,1,Not Selected,0,Room_Type 1,48,Online,0,0,0,94.50,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25524,2,2,0,1,Meal Plan 1,0,Room_Type 6,0,Online,0,0,0,216.00,0,1
25525,3,0,2,6,Meal Plan 1,0,Room_Type 4,85,Online,0,0,0,167.80,1,0
25526,2,0,1,3,Meal Plan 1,0,Room_Type 1,228,Online,0,0,0,90.95,2,1
25527,2,0,2,6,Meal Plan 1,0,Room_Type 1,148,Online,0,0,0,98.39,2,0


In [71]:
df.describe()

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,required_car_parking_space,lead_time,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
count,25529.000000,25529.000000,25529.000000,25529.000000,25529.000000,25529.000000,25529.000000,25529.000000,25529.000000,25529.000000,25529.000000,25529.000000
mean,1.895491,0.143327,0.890869,2.278820,0.042540,67.292334,0.032238,0.029065,0.214971,106.126273,0.749226,0.289866
std,0.527123,0.465622,0.888265,1.516432,0.201821,68.771611,0.176635,0.412627,2.085885,37.772682,0.817069,0.453709
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,0.000000,0.000000,1.000000,0.000000,13.000000,0.000000,0.000000,0.000000,80.750000,0.000000,0.000000
50%,2.000000,0.000000,1.000000,2.000000,0.000000,45.000000,0.000000,0.000000,0.000000,100.300000,1.000000,0.000000
75%,2.000000,0.000000,2.000000,3.000000,0.000000,101.000000,0.000000,0.000000,0.000000,127.500000,1.000000,1.000000
max,4.000000,10.000000,7.000000,17.000000,1.000000,443.000000,1.000000,13.000000,58.000000,540.000000,5.000000,1.000000


Min no. of adults are 0. Which doesn't seem to be accurate...

In [72]:
df[df["no_of_adults"] == 0].shape[0]

134

In [73]:
df = df.drop(df[df["no_of_adults"] == 0].index)
df.shape

(25395, 15)

In [74]:
df = df.reset_index(drop=True)
df

,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,Offline,0,0,0,65.00,0,0
1,2,0,2,3,Not Selected,0,Room_Type 1,5,Online,0,0,0,106.68,1,0
2,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,Online,0,0,0,60.00,0,1
3,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,Online,0,0,0,100.00,0,1
4,2,0,1,1,Not Selected,0,Room_Type 1,48,Online,0,0,0,94.50,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25390,2,2,0,1,Meal Plan 1,0,Room_Type 6,0,Online,0,0,0,216.00,0,1
25391,3,0,2,6,Meal Plan 1,0,Room_Type 4,85,Online,0,0,0,167.80,1,0
25392,2,0,1,3,Meal Plan 1,0,Room_Type 1,228,Online,0,0,0,90.95,2,1
25393,2,0,2,6,Meal Plan 1,0,Room_Type 1,148,Online,0,0,0,98.39,2,0


Everything else seems ok.

---

## Saving the Cleaned Data

In [75]:
df.to_csv("data/cleaned_hotel_reservations_v1.csv", index=False)